# Letterboxd Rating Scraper
Scrapes star ratings for a list of Letterboxd usernames in parallel.

**Output format per user:**
```json
{
  "username": {
    "film_lid": {
      "rating": 4.5,
      "slug": "/film/interstellar/",
      "url": "https://letterboxd.com/film/interstellar/"
    }
  }
}
```

## 1. Install & Imports

In [ ]:
%pip install curl_cffi beautifulsoup4 --quiet

In [ ]:
from curl_cffi.requests import Session as CurlSession
from bs4 import BeautifulSoup as BSoup
import json
import time
import threading
from datetime import datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed

## 2. Config

In [ ]:
LBOXD_URL = "https://letterboxd.com"

# Load from CSV (one username per line, skips header row if present)
with open('usernames.csv') as f:
    rows = [line.strip() for line in f if line.strip()]
USERNAMES = rows[1:] if rows[0].lower() == 'username' else rows
print(f"Loaded {len(USERNAMES)} usernames")
print("First 5:", USERNAMES[:5])

MAX_WORKERS  = 3     # low on purpose -- sharing one IP, Cloudflare rate-limits hard
DELAY        = 1.0   # seconds to sleep between each request within a thread
RETRY_LIMIT  = 3     # retries on 403/5xx before giving up on a page
RETRY_WAIT   = 5.0   # seconds to wait before retrying after a 403
OUTPUT_PATH  = "ratings_output.json"

## 3. Per-thread Session Factory

`curl_cffi` impersonates Chrome's TLS fingerprint at the socket level, which is what Cloudflare actually checks. Each worker thread gets its own session so they don't share state or interfere with each other.

In [ ]:
# Thread-local storage: each thread keeps its own curl_cffi session
_local = threading.local()

def get_session() -> CurlSession:
    """Returns a per-thread curl_cffi session, creating one if needed."""
    if not hasattr(_local, 'session'):
        _local.session = CurlSession(impersonate="chrome124")
        # Warm up this thread's session with a homepage hit
        _local.session.get(LBOXD_URL, timeout=30)
    return _local.session

# Quick test on main thread
print("Testing session...")
resp = get_session().get(LBOXD_URL, timeout=30)
print(f"Status: {resp.status_code} -- {'OK' if resp.status_code == 200 else 'FAILED'}")

## 4. Scraper Functions

In [ ]:
def fetch_with_retry(url: str) -> bytes | None:
    """
    Fetches a URL using the thread-local session.
    Retries up to RETRY_LIMIT times on 403/5xx with exponential backoff.
    Returns response content bytes, or None on failure.
    """
    session = get_session()
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            time.sleep(DELAY)  # polite delay before every request
            resp = session.get(url, allow_redirects=True, timeout=30)
            if resp.status_code == 200:
                return resp.content
            elif resp.status_code in (403, 429, 500, 502, 503):
                wait = RETRY_WAIT * attempt  # 5s, 10s, 15s
                print(f"  [{resp.status_code}] {url} -- retrying in {wait}s (attempt {attempt}/{RETRY_LIMIT})")
                time.sleep(wait)
            else:
                print(f"  [{resp.status_code}] {url} -- skipping")
                return None
        except Exception as e:
            print(f"  [error] {url} -- {e}")
            time.sleep(RETRY_WAIT * attempt)
    return None

In [ ]:
def get_ratings_for_user(username: str) -> dict | None:
    """
    Scrapes all star ratings for a given Letterboxd username.
    Returns {film_lid: {"rating": float, "slug": str, "url": str}}
    or None if the user has no ratings or all requests fail.
    """
    base_url = f"{LBOXD_URL}/{username}/films/ratings/"

    # Page 1 -- also determines total page count
    content = fetch_with_retry(base_url)
    if content is None:
        return None

    soup1 = BSoup(content, "html.parser")

    # Determine total pages
    pagin = soup1.find("div", {"class": "paginate-pages"})
    if pagin is None:
        total_pages = 1
    else:
        items = pagin.find_all("li")
        total_pages = int(items[-1].find("a").text)

    # Fetch remaining pages
    soups = [soup1]
    for page_num in range(2, total_pages + 1):
        url = f"{base_url}page/{page_num}/"
        content = fetch_with_retry(url)
        if content is not None:
            soups.append(BSoup(content, "html.parser"))

    # Parse ratings from all pages
    ratings = {}
    for soup in soups:
        section = soup.find("section", {"class": "section col-main overflow"})
        if section is None:
            continue
        ul = section.find("ul", {"class": "grid"})
        if ul is None:
            continue

        for li in ul.find_all("li"):
            try:
                div = li.find("div", {"class": "react-component"})

                # film ID is JSON inside data-postered-identifier: {"lid": "KzPa", ...}
                import json as _json
                postered = _json.loads(div["data-postered-identifier"])
                film_lid  = postered["lid"]
                film_slug = div["data-target-link"]  # e.g. "/film/interstellar/"

                p = li.find("p")
                rating_span = p.find("span", {"class": lambda c: c and any(cls.startswith("rated-") for cls in c)})
                if rating_span is None:
                    continue  # unrated entry, skip
                rating_class = rating_span["class"]
                # class includes "rated-N" where N is 1-10 (half-star steps)
                rated_val = next(cls for cls in rating_class if cls.startswith("rated-"))
                rating_10 = int(rated_val.split("-")[1])
                rating = rating_10 / 2.0  # convert to 0.5-5.0 star scale

                ratings[film_lid] = {
                    "rating": rating,
                    "slug":   film_slug,
                    "url":    LBOXD_URL + film_slug,
                }
            except Exception:
                continue

    return ratings if ratings else None

In [ ]:
def scrape_users_parallel(
    usernames: list[str],
    max_workers: int = MAX_WORKERS,
    output_path: str = OUTPUT_PATH
) -> dict:
    """
    Scrapes ratings for all usernames in parallel using threads.
    Saves results to JSON after every 50 users so progress isn't lost.
    Returns {username: {film_lid: {"rating", "slug", "url"}}}.
    """
    all_results = {}
    total = len(usernames)
    print(f"[{dt.now()}] Starting scrape for {total} users ({max_workers} workers)\n")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_user = {
            executor.submit(get_ratings_for_user, u): u
            for u in usernames
        }

        for i, future in enumerate(as_completed(future_to_user), 1):
            username = future_to_user[future]
            try:
                ratings = future.result()
                if ratings:
                    all_results[username] = ratings
                    print(f"  [{i}/{total}] {username}: {len(ratings)} ratings")
                else:
                    print(f"  [{i}/{total}] {username}: no ratings found")
            except Exception as e:
                print(f"  [{i}/{total}] {username}: ERROR -- {e}")

            # Save progress every 50 users so you don't lose everything on a crash
            if i % 50 == 0:
                with open(output_path, "w") as f:
                    json.dump(all_results, f, indent=2)
                print(f"  -- checkpoint saved ({len(all_results)} users so far) --")

    # Final save
    with open(output_path, "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"\n[{dt.now()}] Done. {len(all_results)}/{total} users saved to '{output_path}'")
    return all_results

## 5. Run

In [ ]:
results = scrape_users_parallel(USERNAMES)

## 6. Quick Peek at Results

In [ ]:
# Show the first 3 ratings for each user
for user, ratings in results.items():
    print(f"\n{user} ({len(ratings)} ratings):")
    for film_lid, info in list(ratings.items())[:3]:
        print(f"  {film_lid}  {info['rating']}  {info['url']}")